# How `ta_tools` wraps its primitives

`src/tools/ta_tools/` mostly does not reimplement indicators. It calls TA-Lib, or pandas_ta
where TA-Lib has nothing, through thin wrappers — and writes its own only where Pine's
behaviour must be matched exactly and neither library does (`atr` is the example here). This
notebook shows what the wrappers add:

1. **One import site** — `backend.py` is the only module that touches either library.
2. **One contract** — every primitive takes a Series and returns a named float Series on
   the same index, with NaN only in the warm-up.
3. **Pine behaviour where the library differs** — checked against Pine's definitions,
   written out independently below.
4. **Provenance** — `CAPABILITIES` records where each primitive comes from.

Runs offline on seeded bars.


In [1]:
# ── Environment setup: Google Colab vs local ────────────────────────
# Locally the project is installed once from the repo root with `pip install -e ".[ta]"`.
# On Colab the install runs here, and the repo also goes on sys.path because an editable
# install is only picked up when the interpreter starts.
import os
import sys

REPO = '/content/drive/MyDrive/github/quant_dev'

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    if not os.path.isdir(REPO):
        raise FileNotFoundError(f'No repo at {REPO}. Update REPO above.')
    !pip install -q -e "{REPO}[ta]"
    if REPO not in sys.path:
        sys.path.insert(0, REPO)

print(f'Running in Colab: {IN_COLAB}')

Running in Colab: False


## 1. One import site

Importing `ta_tools` loads TA-Lib, which everything is built on, but **not** pandas_ta.
That is loaded the first time a pandas_ta-backed primitive is called, so the beta never
runs unless you use it.


In [2]:
import inspect
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from src.tools import ta_tools
from src.tools.ta_tools import backend

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)

print('public API                        :', ta_tools.__all__)
print('pandas_ta loaded after the import :', 'pandas_ta' in sys.modules)

ta_tools.hma(pd.Series(np.arange(50.0)), 16)   # the first pandas_ta-backed call
print('pandas_ta loaded after first hma  :', 'pandas_ta' in sys.modules)

public API                        : ['CAPABILITIES', 'make_bars', 'sma', 'ema', 'wma', 'hma', 'alma', 'stdev', 'true_range', 'atr', 'bb', 'rsi', 'linreg', 'rma', 'pivot_high', 'pivot_low', 'change', 'crossover', 'crossunder', 'barssince', 'nz', 'recurse']
pandas_ta loaded after the import : False


pandas_ta loaded after first hma  : True


In [3]:
# The whole of what backend.py exposes to the rest of the package.
print(inspect.getsource(backend.as_float_series))
print(inspect.getsource(backend.pandas_ta))

def as_float_series(values):
    """Reject non-Series input and cast to float64, the only dtype TA-Lib accepts."""
    if not isinstance(values, pd.Series):
        raise TypeError(f'expected a pandas Series, got {type(values).__name__}')
    return values.astype('float64')

def pandas_ta():
    """The pandas_ta module, imported on first use so importing ta_tools never loads the beta."""
    try:
        import pandas_ta as module
    except ImportError as exc:
        raise ImportError(
            "This indicator needs pandas_ta; from the repo root run: pip install -e '.[ta]'"
        ) from exc
    return module



## 2. The fixture

`make_bars()` gives seeded OHLCV bars. High and low are built around the candle body, so
`low <= open, close <= high` holds on every bar.


In [4]:
BARS = ta_tools.make_bars(n=400, seed=11)
OPEN, HIGH, LOW, CLOSE = (BARS[c] for c in ('open', 'high', 'low', 'close'))

valid = ((BARS.high >= BARS[['open', 'close']].max(axis=1)) &
         (BARS.low <= BARS[['open', 'close']].min(axis=1))).all()
print(f'{len(BARS)} bars, {BARS.index[0]:%Y-%m-%d} to {BARS.index[-1]:%Y-%m-%d}, all valid OHLC: {valid}')
BARS.head()

400 bars, 2023-01-02 to 2024-07-12, all valid OHLC: True


,open,high,low,close,volume
date,,,,,
2023-01-02,100.180228,100.978120,99.693136,100.077612,3968.0
2023-01-03,99.871116,102.292008,99.447897,101.614526,5574.0
2023-01-04,101.766267,103.383422,101.679185,103.024116,1157.0
2023-01-05,102.965088,103.015452,102.078145,102.487012,9186.0
2023-01-06,102.258240,102.505377,101.272089,102.192089,4803.0


In [5]:
# Chart styling shared by every figure below. Price recedes to greys, leaving colour for the
# indicators, which take a validated categorical palette in fixed order.
SERIES = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4']   # blue, orange, aqua, yellow, magenta
CANDLES = dict(increasing_line_color='#b5b3aa', increasing_fillcolor='#e1e0d9',
               decreasing_line_color='#6f6d68', decreasing_fillcolor='#a9a79f')


def style(fig, title=None, height=480):
    fig.update_layout(title=title, height=height, plot_bgcolor='white', paper_bgcolor='white',
                      hovermode='x unified', xaxis_rangeslider_visible=False,
                      legend=dict(traceorder='normal'), font=dict(color='#52514e'),
                      margin=dict(l=40, r=20, t=50, b=40))
    fig.update_xaxes(gridcolor='#e1e0d9', linecolor='#c3c2b7')
    fig.update_yaxes(gridcolor='#e1e0d9', linecolor='#c3c2b7')
    return fig

## 3. What a wrapper adds

Here is `sma`, whole. The wrapper checks and casts the input, handles the one case TA-Lib
refuses, and names the output. Everything else is TA-Lib.


In [6]:
print(inspect.getsource(ta_tools.sma))

@provides('talib')
def sma(series, length):
    """Simple moving average; the first length-1 values are NaN."""
    series = as_float_series(series)
    if length == 1:  # TA-Lib rejects length 1; Pine returns the input unchanged.
        return series.rename(f'sma_{length}')
    return talib.SMA(series, timeperiod=length).rename(f'sma_{length}')



In [7]:
# Raw TA-Lib against the wrapper, on the same input.
raw = backend.talib.SMA(CLOSE, timeperiod=20)
wrapped = ta_tools.sma(CLOSE, 20)

pd.DataFrame({
    'raw TA-Lib': [type(raw).__name__, raw.name, str(raw.dtype), raw.index.equals(CLOSE.index)],
    'ta_tools.sma': [type(wrapped).__name__, wrapped.name, str(wrapped.dtype),
                     wrapped.index.equals(CLOSE.index)],
}, index=['returns', 'name', 'dtype', 'index kept'])

,raw TA-Lib,ta_tools.sma
returns,Series,Series
name,None,sma_20
dtype,float64,float64
index kept,True,True


In [8]:
# Values are untouched: the wrapper adds a name, not a different calculation.
print('identical values:', np.array_equal(raw.to_numpy(), wrapped.to_numpy(), equal_nan=True))

# And the input check: TA-Lib would accept a bare array and return a bare array.
try:
    ta_tools.sma(CLOSE.to_numpy(), 20)
except TypeError as exc:
    print('ndarray input ->', exc)

identical values: True
ndarray input -> expected a pandas Series, got ndarray


## 4. The contract, across every primitive

Every primitive returns a float Series on the input's index, named `<primitive>_<length>`,
with NaN only in the leading warm-up. The warm-up lengths differ by primitive, and are
part of the contract.


In [9]:
def leading_nans(s):
    return int(np.argmax(s.notna().to_numpy()))


CALLS = {
    'sma':   lambda: ta_tools.sma(CLOSE, 20),
    'ema':   lambda: ta_tools.ema(CLOSE, 20),
    'wma':   lambda: ta_tools.wma(CLOSE, 20),
    'rsi':   lambda: ta_tools.rsi(CLOSE, 14),
    'stdev': lambda: ta_tools.stdev(CLOSE, 20),
    'atr':   lambda: ta_tools.atr(HIGH, LOW, CLOSE, 14),
    'hma':   lambda: ta_tools.hma(CLOSE, 16),
    'alma':  lambda: ta_tools.alma(CLOSE, 9),
}

rows = []
for label, call in CALLS.items():
    out = call()
    warm = leading_nans(out)
    rows.append({
        'primitive': label,
        'source': ta_tools.CAPABILITIES[label],
        'name': out.name,
        'dtype': str(out.dtype),
        'index kept': out.index.equals(CLOSE.index),
        'warm-up NaNs': warm,
        'gaps after warm-up': int(out.iloc[warm:].isna().sum()),
    })
pd.DataFrame(rows).set_index('primitive')

,source,name,dtype,index kept,warm-up NaNs,gaps after warm-up
primitive,,,,,,
sma,talib,sma_20,float64,True,19,0
ema,talib,ema_20,float64,True,19,0
wma,talib,wma_20,float64,True,19,0
rsi,talib,rsi_14,float64,True,14,0
stdev,talib,stdev_20,float64,True,19,0
atr,custom,atr_14,float64,True,13,0
hma,pandas_ta,hma_16,float64,True,18,0
alma,pandas_ta,alma_9,float64,True,8,0


`bb` is the one primitive with three outputs, so it returns a DataFrame with named columns
instead of a Series — which joins straight onto the bars:


In [10]:
bands = ta_tools.bb(CLOSE, 20, 2.0)
print('columns :', list(bands.columns))
print('warm-up :', {c: leading_nans(bands[c]) for c in bands})
BARS.join(bands).iloc[18:23]

columns : ['bb_mid_20', 'bb_upper_20', 'bb_lower_20']
warm-up : {'bb_mid_20': 19, 'bb_upper_20': 19, 'bb_lower_20': 19}


,open,high,low,close,volume,bb_mid_20,bb_upper_20,bb_lower_20
date,,,,,,,,
2023-01-26,104.532370,104.883893,104.202748,104.366384,6971.0,NaN,NaN,NaN
2023-01-27,103.940797,106.261154,103.669486,105.195333,1215.0,102.923738,105.401319,100.446158
2023-01-30,105.184027,105.367882,103.896924,104.230298,3367.0,103.131373,105.296388,100.966357
2023-01-31,103.824395,104.643358,102.364260,102.535699,9962.0,103.177431,105.248568,101.106295
2023-02-01,102.582468,103.199369,102.453851,103.022210,2495.0,103.177336,105.248501,101.106171


## 5. Checked against Pine

Pine's definitions, written out below independently of both libraries, and compared with
each wrapper.


In [11]:
def pine_seeded(values, length, alpha):
    """Pine's ta.ema / ta.rma: SMA of the first `length` values, then the alpha recursion."""
    values = np.asarray(values, dtype=float)
    out = np.full(len(values), np.nan)
    start = int(np.flatnonzero(~np.isnan(values))[0])
    first = start + length - 1
    out[first] = values[start:first + 1].mean()
    for i in range(first + 1, len(values)):
        out[i] = alpha * values[i] + (1 - alpha) * out[i - 1]
    return out


def pine_true_range(high, low, close):
    """Pine's ta.tr(true): on the first bar there is no previous close, so it is high - low."""
    h, l, prev = high.to_numpy(), low.to_numpy(), close.shift(1).to_numpy()
    return np.where(np.isnan(prev), h - l,
                    np.maximum.reduce([h - l, np.abs(h - prev), np.abs(l - prev)]))


def max_diff(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    both = ~np.isnan(a) & ~np.isnan(b)
    return float(np.max(np.abs(a[both] - b[both])))


def pine_rsi(close, length):
    """Pine's RSI: rma of up and down moves."""
    change = close.diff().to_numpy()
    up = np.where(np.isnan(change), np.nan, np.maximum(change, 0))
    down = np.where(np.isnan(change), np.nan, np.maximum(-change, 0))
    return 100 - 100 / (1 + pine_seeded(up, length, 1 / length) / pine_seeded(down, length, 1 / length))


weights = np.arange(1, 21, dtype=float)
pine_wma = CLOSE.rolling(20).apply(lambda w: w @ weights / weights.sum(), raw=True)
pine_basis = CLOSE.rolling(20).mean()
pine_spread = 2.0 * CLOSE.rolling(20).std(ddof=0)
bands = ta_tools.bb(CLOSE, 20, 2.0)

pine_ema = pine_seeded(CLOSE, 20, 2 / 21)
pine_atr = pine_seeded(pine_true_range(HIGH, LOW, CLOSE), 14, 1 / 14)
ours_atr = ta_tools.atr(HIGH, LOW, CLOSE, 14)
talib_atr = backend.talib.ATR(HIGH, LOW, CLOSE, timeperiod=14)

pd.DataFrame([
    {'primitive': 'ema(20)', 'vs': "Pine ta.ema",
     'max abs diff': max_diff(ta_tools.ema(CLOSE, 20), pine_ema),
     'warm-up ours / Pine': f'{leading_nans(ta_tools.ema(CLOSE, 20))} / {leading_nans(pd.Series(pine_ema))}'},
    {'primitive': 'wma(20)', 'vs': 'Pine ta.wma',
     'max abs diff': max_diff(ta_tools.wma(CLOSE, 20), pine_wma), 'warm-up ours / Pine': '19 / 19'},
    {'primitive': 'rsi(14)', 'vs': 'Pine ta.rsi',
     'max abs diff': max_diff(ta_tools.rsi(CLOSE, 14), pine_rsi(CLOSE, 14)), 'warm-up ours / Pine': '14 / 14'},
    {'primitive': 'bb(20, 2) upper', 'vs': 'Pine ta.bb',
     'max abs diff': max_diff(bands['bb_upper_20'], pine_basis + pine_spread), 'warm-up ours / Pine': '19 / 19'},
    {'primitive': 'stdev(20)', 'vs': 'population std (Pine default)',
     'max abs diff': max_diff(ta_tools.stdev(CLOSE, 20), CLOSE.rolling(20).std(ddof=0)),
     'warm-up ours / Pine': '19 / 19'},
    {'primitive': 'stdev(20)', 'vs': "pandas' default rolling std (ddof=1)",
     'max abs diff': max_diff(ta_tools.stdev(CLOSE, 20), CLOSE.rolling(20).std()),
     'warm-up ours / Pine': '—'},
    {'primitive': 'atr(14)', 'vs': 'Pine ta.atr',
     'max abs diff': max_diff(ours_atr, pine_atr),
     'warm-up ours / Pine': f'{leading_nans(ours_atr)} / {leading_nans(pd.Series(pine_atr))}'},
    {'primitive': "TA-Lib's ATR(14)", 'vs': 'Pine ta.atr  (not used)',
     'max abs diff': max_diff(talib_atr, pine_atr),
     'warm-up ours / Pine': f'{leading_nans(talib_atr)} / {leading_nans(pd.Series(pine_atr))}'},
]).set_index('primitive')

,vs,max abs diff,warm-up ours / Pine
primitive,,,
ema(20),Pine ta.ema,5.684342e-14,19 / 19
wma(20),Pine ta.wma,1.676881e-12,19 / 19
rsi(14),Pine ta.rsi,2.842171e-14,14 / 14
"bb(20, 2) upper",Pine ta.bb,2.100364e-11,19 / 19
stdev(20),population std (Pine default),1.055511e-11,19 / 19
stdev(20),pandas' default rolling std (ddof=1),1.256595e-01,—
atr(14),Pine ta.atr,0.000000e+00,13 / 13
TA-Lib's ATR(14),Pine ta.atr (not used),1.410551e-02,14 / 13


`ema`, `wma`, `rsi`, `bb` and `stdev` match Pine exactly. One RSI edge case is not in the
table: on a perfectly flat window TA-Lib returns 0, while TradingView's built-in RSI script
returns 100. Flat 14-bar windows are rare on real prices, and TA-Lib's value is kept. The `ddof=1` row is there because that is the easy
mistake: pandas' default rolling std is the sample std, Pine's is the population std.

`atr` matches Pine too, but only because it is built here, as `rma(true_range)`, rather than
wrapped. The last row shows why: Pine counts the first bar's true range as `high - low`, while
TA-Lib has no true range for that bar, so TA-Lib's ATR starts one bar later and is seeded from
a different window. Its gap from Pine is largest at the start and decays away:


In [12]:
relative = pd.Series(np.abs(talib_atr.to_numpy() - pine_atr) / pine_atr, index=CLOSE.index)

display(pd.DataFrame({'bar': [14, 20, 50, 100, 200, 399],
                      "TA-Lib's ATR vs Pine": [f'{relative.iloc[i]:.2e}' for i in (14, 20, 50, 100, 200, 399)]})
        .set_index('bar'))

fig = go.Figure(go.Scatter(x=relative.index, y=relative * 100, mode='lines',
                           name='|TA-Lib - Pine| / Pine', line=dict(color=SERIES[0], width=2)))
style(fig, "ATR(14): TA-Lib's ATR vs Pine, relative difference (%) — why atr is built here", height=380)
fig.update_yaxes(type='log', title='% (log scale)')
fig.show()

,TA-Lib's ATR vs Pine
bar,
14,9.66e-03
20,6.32e-03
50,7.16e-04
100,1.50e-05
200,9.01e-09
399,2.54e-15


## 6. Where the wrapper answers for TA-Lib

TA-Lib rejects a length of 1 for SMA, EMA, WMA, STDDEV, BBANDS and RSI. Pine accepts it, and the Pine scripts'
inputs allow it (LinReg Candles' `signal_length` has a minimum of 1), so the wrappers return
Pine's answer: the input itself for `sma`, `ema` and `wma`, zero for `stdev`, all three bands
on the input for `bb`, and the direction of each change for `rsi`.


In [13]:
for name, raw_call in [('SMA', lambda: backend.talib.SMA(CLOSE, timeperiod=1)),
                       ('EMA', lambda: backend.talib.EMA(CLOSE, timeperiod=1)),
                       ('WMA', lambda: backend.talib.WMA(CLOSE, timeperiod=1)),
                       ('STDDEV', lambda: backend.talib.STDDEV(CLOSE, timeperiod=1)),
                       ('BBANDS', lambda: backend.talib.BBANDS(CLOSE, timeperiod=1)),
                       ('RSI', lambda: backend.talib.RSI(CLOSE, timeperiod=1))]:
    try:
        raw_call()
        print(f'talib.{name}(length=1): ok')
    except Exception as exc:
        print(f'talib.{name}(length=1): {exc}')

print()
print('ta_tools.sma(close, 1) == close :', ta_tools.sma(CLOSE, 1).equals(CLOSE.rename('sma_1')))
print('ta_tools.ema(close, 1) == close :', ta_tools.ema(CLOSE, 1).equals(CLOSE.rename('ema_1')))
print('ta_tools.stdev(close, 1) all 0  :', bool((ta_tools.stdev(CLOSE, 1) == 0).all()))
print('ta_tools.wma(close, 1) == close :', ta_tools.wma(CLOSE, 1).equals(CLOSE.rename('wma_1')))
print('ta_tools.bb(close, 1) bands     :', all(np.array_equal(b, CLOSE) for _, b in ta_tools.bb(CLOSE, 1).items()),
      '(all three equal the input)')
rsi1, change = ta_tools.rsi(CLOSE, 1), CLOSE.diff()
print('ta_tools.rsi(close, 1)          :', bool((rsi1[change > 0] == 100).all() and (rsi1[change <= 0] == 0).all()),
      '(100 on an up bar, 0 otherwise)')

talib.SMA(length=1): TA_SMA function failed with error code 2: Bad Parameter (TA_BAD_PARAM)
talib.EMA(length=1): TA_EMA function failed with error code 2: Bad Parameter (TA_BAD_PARAM)
talib.WMA(length=1): TA_WMA function failed with error code 2: Bad Parameter (TA_BAD_PARAM)
talib.STDDEV(length=1): TA_STDDEV function failed with error code 2: Bad Parameter (TA_BAD_PARAM)
talib.BBANDS(length=1): TA_BBANDS function failed with error code 2: Bad Parameter (TA_BAD_PARAM)
talib.RSI(length=1): TA_RSI function failed with error code 2: Bad Parameter (TA_BAD_PARAM)

ta_tools.sma(close, 1) == close : True
ta_tools.ema(close, 1) == close : True
ta_tools.stdev(close, 1) all 0  : True
ta_tools.wma(close, 1) == close : True
ta_tools.bb(close, 1) bands     : True (all three equal the input)
ta_tools.rsi(close, 1)          : True (100 on an up bar, 0 otherwise)


## 7. The secondary backend

`hma` and `alma` come from pandas_ta, because TA-Lib has neither. (Section 1 showed the
lazy load: pandas_ta was imported by the first `hma` call, not by importing `ta_tools`.)


In [14]:
print(inspect.getsource(ta_tools.alma))

@provides('pandas_ta')
def alma(series, length, offset=0.85, sigma=6.0):
    """Arnaud Legoux moving average, equal to Pine's ta.alma with floor=true."""
    # pandas_ta's own `offset` argument shifts the output; ALMA's offset is its `dist_offset`.
    out = pandas_ta().alma(as_float_series(series), length=length,
                           sigma=float(sigma), dist_offset=float(offset))
    return out.rename(f'alma_{length}')



The comment in `alma` is the reason to wrap rather than call pandas_ta directly. pandas_ta
functions all take an `offset` argument that **shifts the output**, and `alma` also has a
distribution offset, which it calls `dist_offset` and Pine calls `offset`. Passing Pine's
`offset=0.85` to the wrong one silently gives a different series:


In [15]:
pta = backend.pandas_ta()
right = ta_tools.alma(CLOSE, 9, offset=0.85)
slip = pta.alma(CLOSE, length=9, offset=1)   # a plausible slip: this `offset` shifts the output

both = right.notna() & slip.notna()
print('the slip gives a different series  :', not np.allclose(right[both], slip[both]))
print('it is just the output shifted 1 bar:', np.allclose(slip.dropna(), pta.alma(CLOSE, length=9).shift(1).dropna()))

the slip gives a different series  : True
it is just the output shifted 1 bar: True


pandas_ta's `alma` also centres its weights on `floor(offset * (length - 1))`. Pine does
that only when called with `floor=true`; by default it uses the unrounded value. So the
wrapper matches Pine's `floor=true` variant:


In [16]:
def pine_alma(values, length, offset, sigma, floor):
    """TradingView's reference ta.alma, written out independently of pandas_ta."""
    m = np.floor(offset * (length - 1)) if floor else offset * (length - 1)
    s = length / sigma
    weights = np.exp(-((np.arange(length) - m) ** 2) / (2 * s ** 2))
    weights /= weights.sum()
    values = np.asarray(values, dtype=float)
    out = np.full(len(values), np.nan)
    for i in range(length - 1, len(values)):
        out[i] = np.dot(values[i - length + 1:i + 1], weights)
    return out


ours = ta_tools.alma(CLOSE, 9)
print(f'max |ours - Pine floor=true | = {max_diff(ours, pine_alma(CLOSE, 9, 0.85, 6.0, True)):.2e}')
print(f'max |ours - Pine floor=false| = {max_diff(ours, pine_alma(CLOSE, 9, 0.85, 6.0, False)):.2e}')

max |ours - Pine floor=true | = 2.84e-14
max |ours - Pine floor=false| = 1.17e+00


## 8. Provenance

`CAPABILITIES` is filled in by a `@provides(...)` line above each primitive, so it always
matches the code. `talib` and `pandas_ta` are wrappers; `derived` is composed from TA-Lib calls
(`linreg`); `custom` is written here, for Pine behaviour no library reproduces.


In [17]:
pd.Series(ta_tools.CAPABILITIES, name='source').rename_axis('primitive').to_frame()

,source
primitive,
rsi,talib
sma,talib
ema,talib
wma,talib
hma,pandas_ta
alma,pandas_ta
linreg,derived
rma,custom
pivot_high,custom


In [18]:
fig = go.Figure()
fig.add_trace(go.Candlestick(x=BARS.index, open=OPEN, high=HIGH, low=LOW, close=CLOSE,
                             name='bars', **CANDLES))
AVERAGES = {'sma 20': ta_tools.sma(CLOSE, 20), 'ema 20': ta_tools.ema(CLOSE, 20),
            'wma 20': ta_tools.wma(CLOSE, 20), 'hma 16': ta_tools.hma(CLOSE, 16),
            'alma 9': ta_tools.alma(CLOSE, 9)}
for colour, (label, series) in zip(SERIES, AVERAGES.items()):
    fig.add_trace(go.Scatter(x=series.index, y=series, mode='lines', name=label,
                             line=dict(color=colour, width=2)))
style(fig, 'The five moving averages on the fixture')
fig.show()

The same five series for the last few bars, as a table. Three of the chart's colours (aqua,
yellow, magenta) are below 3:1 contrast against white, so the values are given here too.


In [19]:
pd.concat({'close': CLOSE, **AVERAGES}, axis=1).tail(5).round(2)

,close,sma 20,ema 20,wma 20,hma 16,alma 9
date,,,,,,
2024-07-08,135.36,137.15,136.38,136.92,135.54,135.82
2024-07-09,135.95,137.16,136.34,136.81,135.36,135.73
2024-07-10,138.35,137.34,136.53,136.92,135.61,136.01
2024-07-11,138.94,137.42,136.76,137.07,136.20,136.68
2024-07-12,139.66,137.54,137.04,137.28,137.09,137.64


Bollinger Bands and RSI against the same price, sharing a date axis: the bands measure how
far price sits from its 20-bar mean in standard deviations, RSI how one-sided the recent moves
have been. Touches of the outer bands and RSI extremes tend to line up, but not always.


In [20]:
from plotly.subplots import make_subplots

bands = ta_tools.bb(CLOSE, 20, 2.0)
strength = ta_tools.rsi(CLOSE, 14)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.68, 0.32],
                    vertical_spacing=0.04,
                    subplot_titles=('Bollinger Bands (20, 2)', 'RSI (14)'))

fig.add_trace(go.Candlestick(x=BARS.index, open=OPEN, high=HIGH, low=LOW, close=CLOSE,
                             name='bars', **CANDLES), row=1, col=1)
fig.add_trace(go.Scatter(x=bands.index, y=bands['bb_upper_20'], mode='lines', name='bb upper',
                         line=dict(color=SERIES[0], width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=bands.index, y=bands['bb_lower_20'], mode='lines', name='bb lower',
                         line=dict(color=SERIES[0], width=2), fill='tonexty',
                         fillcolor='rgba(42, 120, 214, 0.08)'), row=1, col=1)
fig.add_trace(go.Scatter(x=bands.index, y=bands['bb_mid_20'], mode='lines', name='bb mid',
                         line=dict(color=SERIES[0], width=2, dash='dot')), row=1, col=1)

fig.add_trace(go.Scatter(x=strength.index, y=strength, mode='lines', name='rsi 14',
                         line=dict(color=SERIES[1], width=2)), row=2, col=1)
for level in (70, 30):
    fig.add_hline(y=level, line=dict(color='#898781', width=1, dash='dash'), row=2, col=1)

style(fig, height=680)
fig.update_yaxes(range=[0, 100], row=2, col=1)
fig.update_xaxes(rangeslider_visible=False)
fig.show()